### AgriPredict: Maize Yield Forecasting for SDG 2

 **Goal**: Predict district-level maize yield in sub-Saharan Africa using open geospatial data.
**Data**: Synthetic but realistic (200 districts, 2018–2022)
**Model**: Random Forest Regressor
**Eval**: MAE, R², SHAP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import shap


In [ ]:
# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("tab10")

In [ ]:
# Load data
df = pd.read_csv("../data/processed/maize_yield_africa.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Define features & target
feature_cols = [
    'ndvi_peak',
    'rain_cum_60d',
    'temp_mean',
    'soil_ph',
    'soil_organic_carbon',
    'elevation',
    'slope',
    'planting_doy'
]
X = df[feature_cols]
y = df['yield_tonnes_per_ha']

print("Features:", feature_cols)
print("Target: yield_tonnes_per_ha")

In [ ]:
# Train-test split (spatially stratified by country)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=df['country']  # Prevent country leakage
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# Train model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("✅ Model trained.")

In [ ]:
# Predict & evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"🎯 MAE: {mae:.2f} tonnes/ha")
print(f"🎯 R²:  {r2:.2f}")

In [ ]:
# Plot: Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, s=80)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Yield (tonnes/ha)")
plt.ylabel("Predicted Yield (tonnes/ha)")
plt.title(f"AgriPredict: Actual vs Predicted (MAE={mae:.2f}, R²={r2:.2f})")
plt.grid(True, alpha=0.3)
plt.savefig("../screenshots/model_performance.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
importances = model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_imp, x='importance', y='feature')
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Explainability
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig("../screenshots/shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print("✅ SHAP plot saved to screenshots/shap_summary.png")

In [ ]:
# Save model
import joblib
joblib.dump(model, "../models/rf_agripredict.pkl")
print("✅ Model saved to models/rf_agripredict.pkl")